# Pneumonia Detection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import pathlib as Path
import os

In [ ]:
gpu = False

# verify if GPU parallel calculus is possible
if(torch.cuda.is_available()):
    print(f"GPU available: {torch.cuda.get_device_name()}")
    gpu = True
else:
    print("No GPU available")

In [ ]:
import shutil
import kagglehub

dataset_url = "paultimothymooney/chest-xray-pneumonia"
LOCAL_DIR = "chest_xray/chest_xray"
KAGGLE_DIR = "/kaggle/input/chest-xray-pneumonia/chest_xray" #kaggle directory for training
DATA_DIR = ""

def load_data():
    """
    Load original pneumonia images data from the Kaggle's dataset. 
    """
    # if it finds the data does nothing otherwise it installs it --> cannot pass the dataset to Github for dimension limitations
    if os.path.exists(KAGGLE_DIR):
        DATA_DIR = KAGGLE_DIR
        print(f"Kaggle data in: {DATA_DIR}")
    elif os.path.exists(LOCAL_DIR):
        DATA_DIR = LOCAL_DIR
        print(f"Local data in: {DATA_DIR}")
    else:
        # Download latest version --> kagglehub stores the dataset in a sort of cached memory so you need to copy it in local
        path = kagglehub.dataset_download(dataset_url)
        
        shutil.copytree(path, ".", dirs_exist_ok=True)

        print("Path to dataset files:", path)
    return DATA_DIR
        
DATA_DIR = load_data()
print(DATA_DIR) 

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, ConcatDataset

RESIZE_IMG = (128, 128) # resize img from (224,224) -> (128, 128)

# Create base transform without augmentations
base_transform = transforms.Compose([
    transforms.Resize(RESIZE_IMG),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load data from train, val, and test folders
train_folder = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), base_transform)
val_folder = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), base_transform)
test_folder = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), base_transform)

# Combine all datasets into one
full_dataset = ConcatDataset([train_folder, val_folder, test_folder])

print(f"Total images: {len(full_dataset)}")

# Split into 80% train and 20% test
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size],
                                           generator=torch.Generator().manual_seed(42))

# Create dictionary with only train and test
image_datasets = {
    "train": train_dataset,
    "test": test_dataset
}

datasets_names = ["train", "test"]
class_names = train_folder.classes
print(class_names)
print(f"Train: {len(train_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# data on the sizes of the single datasets 
dataset_sizes = {x: len(image_datasets[x]) for x in datasets_names}

# printing the number of elements for each dataset.
for x, y in dataset_sizes.items():
    print(f"Dataset {x} has {y} elements.")

In [ ]:
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

def denormalize(tensor):
    """ 
        Given a tensor in input this function returns the corresponding image
        This function is not necessary if in the transformations we do not apply normalization
    """
    img = tensor.numpy().transpose((1, 2, 0)) # change the order of variables in order to get (width, heigth, channels)
    img = std * img + mean
    img = np.clip(img, 0, 1)
    
    return img

# access the first ten images of the train dataset and print it
fig, axis = plt.subplots(2, 5, figsize=(20, 8))
axis = axis.flatten()  # make it an array instead of a matrix
for i in range(10):
    image_tensor, img_label = image_datasets["train"][i]
    
    img = denormalize(image_tensor)
    class_name = class_names[img_label] # if 0 will be 'NORMAL', IF 1 'PNEUMONIA'
    
    axis[i].imshow(img)
    axis[i].set_title(f"Class: {class_name}")
    
    

In [ ]:
BATCH_SIZE = 32
# initializing dataloaders for parallel calculus with GPU
dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    for x in datasets_names
}
# we access images in batches with an iterator
batch_images, batch_labels = next(iter(dataloaders["train"]))

print(batch_images.shape) # prints [batches, channels, width, heigth]
print(len(batch_images))

# another plot of the images, this time shuffled (lo sto facendo solo per prenderci la mano con i plot)
fig, axs = plt.subplots(4, 8, figsize=(20,12))
axs = axs.flatten()
for i in range(BATCH_SIZE):
    img = denormalize(batch_images[i])
    label_name = class_names[batch_labels[i]]
    axs[i].imshow(img)
    axs[i].set_title(f"{label_name}")
    axs[i].axis("off")
    

In [ ]:
# here I define a list of augumentation (I just specified the most important one cited in the paper)
data_augumentations = [
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2), #setted to 20% as the paper suggest
    transforms.RandomAffine(degrees=0, scale=(0.9, 1.1))
]

# here I update the train dataset with the augumentations
data_transforms["train"] = transforms.Compose([
    transforms.Resize(RESIZE_IMG), 
    *data_augumentations, # unpack operators takes the element of the list and puts them inside another one
    transforms.Grayscale(num_output_channels=3), 
    transforms.ToTensor(), 
    transforms.Normalize([0.485, 0.456, 0.406],
                            [0.229, 0.224, 0.225])
])

image_datasets["train"] = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), data_transforms["train"]) # just modify the train one

# update dataloaders with the new augmented set
dataloaders["train"] = DataLoader(image_datasets["train"], batch_size=BATCH_SIZE, shuffle=True, num_workers=2)



In [ ]:
batch_images, batch_labels = next(iter(dataloaders["train"]))

# last images plot ( I SET THE BRIGHTEN AUGUMENTATION TO 50% JUST TO MAKE THE RESULTS VISIBLE BUT THAT HAS TO BE CHANGED TO 20%)
fig, axs = plt.subplots(4, 8, figsize=(20,12))
axs = axs.flatten()
for i in range(BATCH_SIZE):
    img = denormalize(batch_images[i])
    label_name = class_names[batch_labels[i]]
    axs[i].imshow(img)
    axs[i].set_title(f"{label_name}")
    axs[i].axis("off")


In [ ]:
#apply oversampling for class balance with SMOTE tecnique

from imblearn.over_sampling import SMOTE 
from torch.utils.data import TensorDataset

#count occurrences
train_label = image_datasets['train'].targets
count_normal = train_label.count(0)
count_pneumonia = train_label.count(1)

print(f"Count Normal: {count_normal}")
print(f"Count Pneumonia: {count_pneumonia}")

#we have to flatter images because SMOTE works with "tables" of numbers--> so we have to transform images into number vectors 

#temporary lists
train_data_flat = []
train_labels = []

for img, label in image_datasets['train']:
    train_data_flat.append(img.numpy().flatten()) #numpy transform img in array, flattern transorm it in one dim array
    train_labels.append(label)

train_data_flat = np.array(train_data_flat)
train_labels = np.array(train_labels)

print(f"Actual dimension: {train_data_flat.shape}")

smote = SMOTE(random_state=42)

img_resampled, label_resampled = smote.fit_resample(train_data_flat, train_labels) #fit resample learn from datas and make new datas similar 

print(f"Now we have: {len(label_resampled)} images.")

#check the balance
unique, counts = np.unique(label_resampled, return_counts=True)
print(counts)

#at the end we have to re-convert arrays of number in images for ur CNN

img_resampled_tensor = torch.tensor(img_resampled).float().view(-1, 3, 128, 128) 
label_resampled_tensor = torch.tensor(label_resampled).long()

train_dataset_smote = TensorDataset(img_resampled_tensor, label_resampled_tensor)

dataloaders["train"] = DataLoader(train_dataset_smote, batch_size=BATCH_SIZE, shuffle=True, num_workers=2) # save the SMOTE dataset into a DataLoader

In [ ]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, (3, 3), padding=1, stride=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, (3, 3), padding=1, stride=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2), 

            nn.Conv2d(64, 128, (3, 3), padding=1, stride=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Flatten(), 
            
            # 32768 implies our input images are 128x128.
            nn.Linear(32768, 128),
            nn.Linear(128, 64) # Output size is 64
        )
        
    def forward(self, x):
        return self.cnn(x)

print(batch_images.shape) # print again the tensor shape
cnn = CNN()

In [ ]:
device = torch.device("cuda" if gpu else "cpu") # set device for execution
cnn.to(device)

# eval() changed the mode the BatchNorm is executed in the pipeline: ...
cnn.eval()

features = [] # define an array that will contain all the batches

# iterate over all batches and apply the pipeline
for batch_idx, (images, labels) in enumerate(dataloaders["train"]):
    images = images.to(device) # move data to CPU/GPU
    
    with torch.no_grad():
        output = cnn(images) # apply the pipeline to the image
    features.append(output.cpu()) #avoid gpu to overflow
    
    # print every ten batches
    if (batch_idx + 1) % 10 == 0:
        print(f"Batch n.{batch_idx + 1}: Processed.")
        
print(features[0].shape) # print the shape of the first element of features after applying the transformation

In [ ]:
num_steps = 25

class BiGRU(nn.Module):
    def __init__(self, num_steps, input_size=64, hidden_size=256, dropout=0.3):
        super().__init__()
        
        self.num_steps = num_steps
        self.bigru = nn.GRU(
                    input_size=input_size,
                    hidden_size=hidden_size,
                    num_layers=2,
                    dropout=dropout,
                    bidirectional=True,
                    batch_first=True
        )
        
    def forward(self, x):
        """
            x: Input tensor of shape [batch_size, input_size]
            Returns an output tensor of shape (batch_size, num_steps, hidden_size * 2)
        """
        x_seq = x.unsqueeze(1).repeat(1, self.num_steps, 1) # repeat the sample for num_steps times
        
        # out shape: (batch_size, num_steps, hidden_size * 2)
        out, _ = self.bigru(x_seq)
        
        return out

bigru = BiGRU(
    num_steps=num_steps
)


In [ ]:
# I KEEP THIS CODE HERE JUST TO SEE IF EVERYTHING WORKS, WE WILL CANCEL IT EVENTUALLY

bigru_output = [] 

for feature in features:
        out = bigru(feature)
        bigru_output.append(out)
    
print(f"Max input value: {bigru_output[0].max()}")
print(f"Min input value: {bigru_output[0].min()}")
print(f"Mean input value: {bigru_output[0].mean()}")
print(bigru_output[0].shape)
print(bigru_output[0])

In [ ]:
import snntorch as snn
from snntorch import surrogate

class DoubleSNN(nn.Module):
    def __init__(self, input_size=512, hidden_size=256):
        super().__init__()
        beta = 0.95 # setup value of beta (given in the paper)
        
        spike_grad = surrogate.fast_sigmoid(slope=25) # surrogate gradient will be used for backpropagation (used default value here)
        self.fc1 = nn.Linear(input_size, 256) # define the projection to 256 dimension
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True, output=True)
        
        #si potrebbe modificare la treshold a 0.1 se quella impostata a 1 di default è troppo bassa
        concat_size = input_size + hidden_size
        self.fc2 = nn.Linear(concat_size, concat_size)    
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True, output=True) # returns mem and spk if output=True
        
    def forward(self, x):
        """
            Takes in input the tensor with the time_steps defined in the BiGRU layer.
        """
        self.lif1.reset_mem()
        self.lif2.reset_mem()
                
        spk_list = [] # snn results will be saved here 
        
        for step in range(num_steps):
            snn_input = x[:, step, :] # because the bigru output is a tensor like [batch_size, num_steps, 512]
            
            cur1 = self.fc1(snn_input)
            _, mem1 = self.lif1(cur1) # save the state that comes from the hidden membrane
            
            concat_input = torch.cat((mem1, snn_input), dim=1) # concatenate the snn_input with the output of the first SNN layer
            
            cur2 = self.fc2(concat_input)
            spk2, _ = self.lif2(cur2)
            
            spk_list.append(spk2) # store the spike output into a list
            
        # here we stack data along the num_steps dimension. Basically we take all the spk_output for each time step
        # and we put it in an array. In the end we perform spike mean aggregation along the time_step dimension.
        stacked_spk = torch.stack(spk_list, dim=1)
        return torch.mean(stacked_spk, dim=1)
    
spikenn = DoubleSNN()

In [ ]:
# Again I use this block just to verify results of the SNN
snn_output = []

for batch_bigru in bigru_output:
    out = spikenn(batch_bigru)
    snn_output.append(out)
    
print(len(snn_output)) # make sure the spk list is empty now
print(snn_output[0].shape) # aggregated data dimension
    

In [ ]:
# define the sequential pipeline 
decision_head = nn.Sequential(
    nn.Linear(768, 512),
    nn.Dropout(0.4),
    nn.Linear(512, 128),
    nn.Dropout(0.4),
    nn.Linear(128, 2)
)

for b in range(len(snn_output)):
    output = decision_head(snn_output[b])
    
    print(output)


In [ ]:
# definition of the Net

class FullNet(nn.Module):
    def __init__(self, num_steps, decision_head):
        super().__init__()
        
        # initialize the layers defined as classes previously
        self.cnn = CNN()
        self.bigru = BiGRU(
            num_steps=num_steps
        )
        self.snn = DoubleSNN(        )
        self.decision_head = decision_head # initialize the layer which linearizes until [batch_size, num_labels] (2 labels in this case)
        
        
    def forward(self, x):
        """
            x : input of shape [batch_size, channels, img_width, img_heigth]
        """
        # apply all the input across all the layers
        x = self.cnn(x)
        x = self.bigru(x)
        x = self.snn(x)
        
        logits = self.decision_head(x)
        
        return logits
    
model = FullNet(num_steps, decision_head)

TODO: DEFINE CROSS ENTROPY IN MARKDOWN,
DEFINE THE ONECYCLELR 

In [ ]:
import torch.optim as optim

# definition of some usefull parameters
TRAIN_EPOCHS = 30 # number of max training epochs
TARGET_EPOCHS = 7 # actual number of training epochs performed (at least this is said on the paper)

# define some parameters to pass the Adam optimizer
lr = 1e-5 
w_decay = 1e-5

optimizer = optim.Adam(
    model.parameters(),
    lr=lr,
    weight_decay=w_decay
)

steps_per_epoch = len(dataloaders["train"]) # steps per epoch is basically the number of batches inside the dataloader
print(len(dataloaders["train"]))

# define the OneCycleLR scheduler
lr_scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer=optimizer,
    max_lr=lr,
    epochs=TRAIN_EPOCHS,
    steps_per_epoch=steps_per_epoch
)

halving_param = 5 # training halves early if no improvement in 5 consecutive epochs (check paper)

# define the CrossEntropyLoss object. It expects raw logits as the ones given in output by the pipeline
# Applies SoftMax and computes the NLLL loss.
loss = nn.CrossEntropyLoss() 

# (PER NICOLA) controlla documentatione di pytorch però di base prende i logits di una batch e i label che prendi dal dataloader.
# con un for loop iteri su tutte le batch del dataloader e ti prendi anche i label. 


In [ ]:
def eval_model(model, dataloader, loss_fn, device):

    model.eval()  # Set to evaluation mode
    
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():  # Disable gradient computation
        for batch_images, batch_labels in dataloader:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)
            
            # Forward pass
            logits = model(batch_images)
            loss_value = loss_fn(logits, batch_labels)
            
            # Accumulate loss
            total_loss += loss_value.item()
            
            # Calculate accuracy
            predictions = torch.argmax(logits, dim=1) #class with the highest logit
            correct += (predictions == batch_labels).sum().item() #compare predictions with true labels and then sum
            total += batch_labels.size(0)
    
    model.train()  # Set back to training mode
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    
    return avg_loss, accuracy


In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import SubsetRandomSampler

device = torch.device("cuda" if gpu else "cpu")
model.to(device)

count = 0
best_loss = float('inf')
k_folds = 5
batch_size = 32

#Defining the k fold for cross-validation
kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)

#full train tat we have to divide into folders for cross validation
full_train_dataset = dataloaders["train"].dataset

results = {} #to save accuracy of each fold validation

#itarating over folds
for fold, (train_idx, val_idx) in enumerate(kfold.split(full_train_dataset)):
    print(fold, train_idx, val_idx)
    
    #subset for the current fold
    sub_train = SubsetRandomSampler(train_idx)
    sub_val = SubsetRandomSampler(val_idx)
    print(f"sub-train: {sub_train}")


    #dataloader for the current fold
    train_loader = DataLoader(full_train_dataset, batch_size=batch_size, sampler=sub_train)
    val_loader = DataLoader(full_train_dataset, batch_size=batch_size, sampler=sub_val)

    #we have to initialize the model every time to reset and don't corrupt the results
    model = FullNet(num_steps, decision_head) 
    model.to(device)

    #calculate steps per epoch (the size of the train set is changed)
    steps_per_epoch = len(train_loader)

    best_loss = float('inf')
    #start training of the current fold
    for epoch in range(TRAIN_EPOCHS):
        
        model.train()  # Set to training mode
        
        for batch_images, batch_labels in train_loader:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)
            
            # Forward pass
            logits = model(batch_images)
            loss_value = loss(logits, batch_labels)

            # Backward pass
            optimizer.zero_grad()
            loss_value.backward()

            # Update parameters through Adam and OneCycleLR
            optimizer.step()
            lr_scheduler.step()
        
        # Validation after each epoch of the current folder
        val_loss, val_acc = eval_model(model, val_loader, loss, device)
        print(f"Epoch {epoch+1}/{TRAIN_EPOCHS} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

        # Early stop criteria and implementation
        if val_loss < best_loss:
            best_loss = val_loss  # Update best loss
            count = 0  # Reset counter for consecutive epochs without improvements
        else: 
            count += 1

        if count >= halving_param:
            print(f"Early stopping: no improvement in validation loss for {halving_param} consecutive epochs")
            break

    results[fold] = val_acc * 100
    print(f"Fold accuracy {fold}: {val_acc*100:.2f}%")



In [ ]:
test_loss, test_acc = eval_model(model, dataloaders["test"], loss, device)
print('avarage loss: ', test_loss)
print('accuracy', test_acc*100)